In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import geopandas as gpd

Importing county, and state shapefiles and FIPS codes below

In [ ]:
us_fips_code = pd.read_csv("E:/DISES/quantitative_data/ancillary_data/us-state-ansi-fips.csv")
us_fips_code.columns = ['STATE', "STATEFP", "ST"]
us_fips_code["STATE"] = us_fips_code["STATE"].str.upper()

In [ ]:
us_county = gpd.read_file("E:/DISES/quantitative_data/ancillary_data/conus_county.shp")

us_county = us_county[["STATEFP","COUNTYFP", "GEOID", "NAME", "geometry"]]


us_county["STATEFP"] = pd.to_numeric(us_county["STATEFP"])

us_county["NAME"] = us_county["NAME"].str.upper()

us_county = us_county.merge(us_fips_code, left_on = "STATEFP", right_on = "STATEFP").drop(["STATE"], axis = 1)

us_county_geo = us_county.to_crs(4326)

geoid_geometry = us_county[["GEOID", "geometry"]]

In [ ]:
conus = gpd.read_file("E:/DISES/quantitative_data/ancillary_data/conus_states.shp")
conus["NAME"] = conus["NAME"].str.upper()
conus = conus.to_crs(epsg=4326)
conus = conus[[ "STUSPS", "NAME",  "geometry"]]

conus = conus[~conus["STUSPS"].isin(["AS", "GU", "MP"])]

conus =conus.rename(columns={"STUSPS":"ST"})

conus_geometry = conus[["ST", "geometry"]]

In [ ]:
conus.head()

In [ ]:
st_id_east = ["23", "38", "46" ,"55", "50", "27", "33" ,"19" ,"25" ,"31" ,"36" ,"42", "09", "44", "34", "18", "39", "17" ,"11", "10", "54", "24", "21" ,"20" ,"51", "29", "40" ,"37",
"47", "48" ,"01", "28", "13" ,"45", "05", "22", "12", "26"]

In [ ]:
st_id_east_int = [int(i) for i in st_id_east]

In [ ]:
state_list_east = ["ME", "NH", "VT", "MA", "RI", "CT", "NY", "PA", "NJ", "DE", "MD", "WV", "VA", "NC", "SC", "KY",
                "OH", "MI", "WI", "IA", "MN", "MO", "TN", "GA", "AL", "MS", "LA", "AR", "KS", "OK", 
                "TX", "ND", "NE", "SD", "IL", "IN", "FL", "DC"]

In [ ]:
conus_east = conus[conus["ST"].isin(state_list_east)]

_nass_clean_ cleans the Quickstat csv file

In [ ]:
def nass_clean(df, program = "average"):
    df = df[["Program", "Year", "State", "State ANSI", "County ANSI",  "County", "Commodity","Domain", "Domain Category", "Data Item", "Value"]]
    df = df[~df["State"].isin(["ALASKA", "HAWAII"])]
    df["Value"] = df["Value"].astype("string")
    df = df[~df["Value"].str.contains("(D)", regex=False)]
    df = df[~df["Value"].str.contains("(Z)", regex=False)]
    df = df[~df["Value"].str.contains("NA", regex=False)]
    df = df[~df["Value"].str.contains("<NA>", regex=False)]
    df = df[df["County"].notna()]
    df = df[~df["County"].str.contains("OTHER (COMBINED) COUNTIES", regex=False)]    
    df = df[~df["County"].str.contains("OTHER COUNTIES", regex=False)]    
    df["Value"] = df["Value"].replace(',','', regex=True)
    df["Value"] = pd.to_numeric(df["Value"])    
    df["GEOID"] = df["State ANSI"].apply(lambda x: f'{int(x):02}') + \
                    df["County ANSI"].apply(lambda x: f'{int(x):03}')    
    
    if program == "census":
        df = df[df["Program"]=="CENSUS"]
        
    if program == "survey":
        df = df[df["Program"]=="SURVEY"]
        
    if program == "average":
        df = df.groupby(by = ["Year", "State", "State ANSI", "County ANSI",  "County",'GEOID', "Commodity","Domain", "Domain Category",  "Data Item"])["Value"].mean().reset_index()
        df["Program"] = "AVG_CEN_SUR"
        df = df.loc[:, ['Program'] + [col for col in df.columns if col != 'Program']]
   

    if len(df["Commodity"].unique())>1:
        print("More than one Commodity found: ", df["Commodity"].unique())
    if len(df["Domain"].unique())>1:
        print("More than one Domain found: ", df["Domain"].unique())
    if len(df["Domain Category"].unique())>1:
        print("More than one Domain Category found: ", df["Domain Category"].unique())
    if len(df["Data Item"].unique())>1:
        print("More than one Data Item found: ", df["Data Item"].unique())
    df = df.drop("Program", axis = 1)
    return df

# Non crop-specific variables (same for all crops)

In [ ]:
land_practices1 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/land_practices_2017.csv"), program="census")
land_practices2 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/land_practices_2022.csv"), program="census")

In [ ]:
land_practices_all = pd.concat([land_practices1, land_practices2], axis = 0)
land_practices_all = land_practices_all.reset_index(drop = True)

In [ ]:
land_practices_all["Data Item"].unique()

### Cover Crop

In [ ]:
cover_crop_acre = land_practices_all[land_practices_all["Data Item"] == "PRACTICES, LAND USE, CROPLAND, COVER CROP PLANTED, (EXCL CRP) - ACRES"]
cover_crop_acre = cover_crop_acre.reset_index(drop = True)

In [ ]:
cover_crop_acre["Year"].unique()

In [ ]:
cover_crop_acre["Year"].unique()

In [ ]:
cover_crop_acre["Data Item"].unique()

### Tillage Info (conventional, conservational, and no-till)

In [ ]:
till_conventional_acre = land_practices_all[land_practices_all["Data Item"] == "PRACTICES, LAND USE, CROPLAND, CONVENTIONAL TILLAGE - ACRES"]
till_conventional_acre = till_conventional_acre.reset_index(drop = True)

In [ ]:
till_conservational_acre = land_practices_all[land_practices_all["Data Item"] == "PRACTICES, LAND USE, CROPLAND, CONSERVATION TILLAGE, (EXCL NO-TILL) - ACRES"]
till_conservational_acre = till_conservational_acre.reset_index(drop = True)

In [ ]:
till_no_till_acre = land_practices_all[land_practices_all["Data Item"] == "PRACTICES, LAND USE, CROPLAND, CONSERVATION TILLAGE, NO-TILL - ACRES"]
till_no_till_acre = till_no_till_acre.reset_index(drop = True)

In [ ]:
till_conservational_acre["Year"].unique()

In [ ]:
till_conservational_acre

### Irrigated areas total 

In [ ]:
irg_acre_total = nass_clean(pd.read_csv("E:/DISES/quantitative_data/irrigated_acre_harvested_total_cropland.csv"), program="census")

In [ ]:
irg_acre_total

In [ ]:
irg_acre_total["Year"].unique()

### Tile-drainage acres

In [ ]:
tile_drain_acres = nass_clean(pd.read_csv("E:/DISES/quantitative_data/tile_drainage_acres.csv"), program="census")

In [ ]:
tile_drain_acres["Year"].unique()

In [ ]:
tile_drain_acres["Data Item"].unique()

### Organic no of op

In [ ]:
organic_number_operations = nass_clean(pd.read_csv("E:/DISES/quantitative_data/organic_number_operations.csv"), program="census")

In [ ]:
# Both domain category should be added

In [ ]:
organic_number_operations["Data Item"].unique()

In [ ]:
organic_number_operations["Year"].unique()

### Precision Ag Number of operations

In [ ]:
precision_ag_number_operations = nass_clean(pd.read_csv("E:/DISES/quantitative_data/precision_ag_nuber_operations.csv"), program="census")

In [ ]:
precision_ag_number_operations["Year"].unique()

In [ ]:
precision_ag_number_operations["Data Item"].unique()

## Crop insurance acres RMA

In [ ]:
crop_insurance_acres_rma = pd.read_csv("E:/DISES/quantitative_data/crop_insurance_acres_rma.csv")

In [ ]:
crop_insurance_acres_rma["GEOID"] = crop_insurance_acres_rma["GEOID"].astype("string").str.zfill(5)

In [ ]:
crop_insurance_acres_rma = crop_insurance_acres_rma.drop("n", axis = 1)
crop_insurance_acres_rma.columns = ["GEOID", "crop_insurance_acre_rma"]

In [ ]:
crop_insurance_acres_rma["Year"] = 2017

In [ ]:
crop_insurance_acres_rma.head()

### CRP acres and payment data from Farm Service Agency 

In [ ]:
crp_acre_fsa = pd.read_excel("E:/DISES/quantitative_data/CRPHistoryCounty86-22.xlsx", sheet_name="ACRES")
crp_usd_fsa = pd.read_excel("E:/DISES/quantitative_data/CRPHistoryCounty86-22.xlsx", sheet_name="RENT")

In [ ]:
crp_acre_fsa.columns = crp_acre_fsa.iloc[2]
crp_usd_fsa.columns = crp_usd_fsa.iloc[1]

In [ ]:
crp_acre_fsa = crp_acre_fsa[crp_acre_fsa["FIPS"]!=72999]
crp_usd_fsa = crp_usd_fsa[crp_usd_fsa["FIPS"]!=72999]

In [ ]:
crp_acre_fsa = crp_acre_fsa[~crp_acre_fsa["FIPS"].isna()].reset_index(drop = True).iloc[1:]
crp_usd_fsa = crp_usd_fsa[~crp_usd_fsa["FIPS"].isna()].reset_index(drop = True).iloc[1:]

In [ ]:
crp_usd_fsa["GEOID"] = crp_usd_fsa["FIPS"].astype("string").str.zfill(5)
crp_acre_fsa["GEOID"] = crp_acre_fsa["FIPS"].astype("string").str.zfill(5)

In [ ]:
crp_usd_fsa = crp_usd_fsa.drop(["STATE", "COUNTY", "FIPS"], axis = 1).set_index("GEOID")
crp_acre_fsa = crp_acre_fsa.drop(["STATE", "COUNTY", "FIPS"], axis = 1).set_index("GEOID")

In [ ]:
crp_usd_fsa.columns.name =""
crp_acre_fsa.columns.name =""

In [ ]:
crp_usd_fsa_melt = crp_usd_fsa.loc[:,2008:].reset_index().melt(id_vars=['GEOID'], var_name='Year', value_name='Value')
crp_acre_fsa_melt = crp_acre_fsa.loc[:,2008:].reset_index().melt(id_vars=['GEOID'], var_name='Year', value_name='Value')

In [ ]:
crp_usd_fsa_melt["Data Item"] = "CRP_USD"
crp_acre_fsa_melt["Data Item"] = "CRP_ACRE"

In [ ]:
crp_acre_fsa_melt

In [ ]:
crp_acre_fsa_melt["Year"].unique()

### Fertilizers

In [ ]:
fert_org = nass_clean(pd.read_csv("E:/DISES/quantitative_data/fertilizer_expenses.csv"), program="census")

In [ ]:
fert_org["Data Item"].unique()

In [ ]:
fert_exp_usd = fert_org[fert_org["Data Item"]=="FERTILIZER TOTALS, INCL LIME & SOIL CONDITIONERS - EXPENSE, MEASURED IN $"]

In [ ]:
fert_exp_usd

In [ ]:
fert_exp_usd["Year"].unique()

In [ ]:
chem_org = nass_clean(pd.read_csv("E:/DISES/quantitative_data/chemical_expenses.csv"), program="census")

In [ ]:
chem_org["Data Item"].unique()

In [ ]:
chem_exp_usd = chem_org[chem_org["Data Item"]=="CHEMICAL TOTALS - EXPENSE, MEASURED IN $"]

In [ ]:
chem_exp_usd["Year"].unique()

### N and P in kg from Manure and Fertilizer

Download from other than emory IP

In [ ]:
fert_farm_kg = pd.read_csv("E:/DISES/quantitative_data/N&P_from_manure_fert_sciencebase/fert-farm-1987-2017.txt")

In [ ]:
fert_farm_kg["GEOID"] = fert_farm_kg["STCOFIPS"].apply(lambda x: str(x).zfill(5))

In [ ]:
fert_farm_kg.columns

In [ ]:
fert_farm_kg = fert_farm_kg[['GEOID', 'farmfertN-kg-1997', 'farmfertN-kg-2002',
       'farmfertN-kg-2007', 'farmfertN-kg-2012', 'farmfertN-kg-2017',
             'farmfertP-kg-1997',
       'farmfertP-kg-2002', 'farmfertP-kg-2007', 'farmfertP-kg-2012',
       'farmfertP-kg-2017']]

In [ ]:
temp_N = fert_farm_kg[['GEOID', 'farmfertN-kg-1997', 'farmfertN-kg-2002',
       'farmfertN-kg-2007', 'farmfertN-kg-2012', 'farmfertN-kg-2017']]
fert_farm_kg_N = pd.melt(temp_N, id_vars=['GEOID'], 
                    var_name='Year_str', 
                    value_name='N_fert_kg')


fert_farm_kg_N["Year"] = fert_farm_kg_N["Year_str"].str.split("-").str[-1].astype("int")

fert_farm_kg_N = fert_farm_kg_N.drop("Year_str", axis = 1)

In [ ]:
temp_P = fert_farm_kg[['GEOID', 'farmfertP-kg-1997',
       'farmfertP-kg-2002', 'farmfertP-kg-2007', 'farmfertP-kg-2012',
       'farmfertP-kg-2017']]
fert_farm_kg_P = pd.melt(temp_P, id_vars=['GEOID'], 
                    var_name='Year_str', 
                    value_name='P_fert_kg')


fert_farm_kg_P["Year"] = fert_farm_kg_P["Year_str"].str.split("-").str[-1].astype("int")

fert_farm_kg_P = fert_farm_kg_P.drop("Year_str", axis = 1)

In [ ]:
manure_farm_kg_97 = pd.read_csv("E:/DISES/quantitative_data/N&P_from_manure_fert_sciencebase/manure1997.txt")[['STCOFIPS','Total_N_kg-1997', 'Total_P_kg-1997']].set_index("STCOFIPS")
manure_farm_kg_02 = pd.read_csv("E:/DISES/quantitative_data/N&P_from_manure_fert_sciencebase/manure2002.txt")[['STCOFIPS','Total_N_kg-2002', 'Total_P_kg-2002']].set_index("STCOFIPS")
manure_farm_kg_07 = pd.read_csv("E:/DISES/quantitative_data/N&P_from_manure_fert_sciencebase/manure2007.txt")[['STCOFIPS','Total_N_kg-2007', 'Total_P_kg-2007']].set_index("STCOFIPS")
manure_farm_kg_12 = pd.read_csv("E:/DISES/quantitative_data/N&P_from_manure_fert_sciencebase/manure2012.txt")[['STCOFIPS','Total_N_kg-2012', 'Total_P_kg-2012']].set_index("STCOFIPS")
manure_farm_kg_17 = pd.read_csv("E:/DISES/quantitative_data/N&P_from_manure_fert_sciencebase/manure2017.txt")[['STCOFIPS','Total_N_kg-2017', 'Total_P_kg-2017']].set_index("STCOFIPS")

In [ ]:
manure_farm_kg_all = pd.concat([manure_farm_kg_97, manure_farm_kg_02, manure_farm_kg_07, manure_farm_kg_12, manure_farm_kg_17], axis = 1).reset_index()

In [ ]:
manure_farm_kg_all["GEOID"] = manure_farm_kg_all["STCOFIPS"].apply(lambda x: str(x).zfill(5))

In [ ]:
manure_farm_kg_all = manure_farm_kg_all.drop("STCOFIPS", axis = 1)

In [ ]:
manure_farm_kg_all

In [ ]:
temp_N_man = manure_farm_kg_all[['GEOID', 'Total_N_kg-1997', 'Total_N_kg-2002',
       'Total_N_kg-2007', 'Total_N_kg-2012', 'Total_N_kg-2017']]
manure_farm_kg_N = pd.melt(temp_N_man, id_vars=['GEOID'], 
                    var_name='Year_str', 
                    value_name='N_manure_kg')


manure_farm_kg_N["Year"] = manure_farm_kg_N["Year_str"].str.split("-").str[-1].astype("int")

manure_farm_kg_N = manure_farm_kg_N.drop("Year_str", axis = 1)

In [ ]:
temp_P_man = manure_farm_kg_all[['GEOID', 'Total_P_kg-1997', 'Total_P_kg-2002',
       'Total_P_kg-2007', 'Total_P_kg-2012', 'Total_P_kg-2017']]
manure_farm_kg_P = pd.melt(temp_P_man, id_vars=['GEOID'], 
                    var_name='Year_str', 
                    value_name='P_manure_kg')


manure_farm_kg_P["Year"] = manure_farm_kg_P["Year_str"].str.split("-").str[-1].astype("int")

manure_farm_kg_P = manure_farm_kg_P.drop("Year_str", axis = 1)

In [ ]:
NP_from_manure_fert = fert_farm_kg_N.merge(fert_farm_kg_P).merge(manure_farm_kg_N).merge(manure_farm_kg_P)

In [ ]:
NP_from_manure_fert.head()

### Labor expenses

In [ ]:
labor_usd = nass_clean(pd.read_csv("E:/DISES/quantitative_data/labor_hired&contract_exp.csv"), program="census")

In [ ]:
labor_usd["Data Item"].unique()

In [ ]:
labor_usd = labor_usd.drop(["Data Item"], axis = 1).groupby(["Year", "State", "State ANSI","County ANSI", "County", "Commodity", "GEOID"]).sum().reset_index()
labor_usd["Data Item"] = "LABOR, CONTRACT plus HIRED - EXPENSE, MEASURED IN $"

In [ ]:
labor_usd["Year"].unique()

In [ ]:
labor_usd["Data Item"].unique()

In [ ]:
labor_usd[(labor_usd["GEOID"]=="01001") & (labor_usd["Year"] == 2022)]

### Machineary expenses

In [ ]:
machinery_usd = nass_clean(pd.read_csv("E:/DISES/quantitative_data/machinery_usd.csv"), program="census")

In [ ]:
machinery_usd["Data Item"].unique()

In [ ]:
machinery_usd["Year"].unique()

### Median farm size

In [ ]:
median_farm_size = nass_clean(pd.read_csv("E:/DISES/quantitative_data/median_farm_size.csv"))

In [ ]:
median_farm_size.loc[median_farm_size["Data Item"]=='FARM OPERATIONS - AREA OPERATED, MEASURED IN ACRES / OPERATION, MEDIAN', "Data Item"] = "Median Farm Size Acres"

In [ ]:
median_farm_size

In [ ]:
median_farm_size["Year"].unique()

### Livestock

In [ ]:
live_sales_doug_2017 = pd.read_csv("E:/DISES/quantitative_data/F_2017_County_Agcensus_crop_livestock_salestotal_imputed2.csv")

In [ ]:
live_sales_doug_2017 = live_sales_doug_2017[["fips", "Data Item", "Value"]]
live_sales_doug_2017["GEOID"] = live_sales_doug_2017["fips"].astype("str").str.zfill(5)

In [ ]:
live_sales_doug_2017["Data Item"].unique()

In [ ]:
live_sales_doug_2017["Value"] = live_sales_doug_2017["Value"].replace(',','', regex=True)
live_sales_doug_2017["Value"] = pd.to_numeric(live_sales_doug_2017["Value"])   

In [ ]:
live_sales_doug_2017 = live_sales_doug_2017[live_sales_doug_2017["Data Item"]=="ANIMAL TOTALS, INCL PRODUCTS - SALES, MEASURED IN $"]
live_sales_doug_2017 =live_sales_doug_2017.reset_index(drop = True)

In [ ]:
live_sales_doug_2017 = live_sales_doug_2017.pivot_table(values="Value", index = "GEOID",columns="Data Item")

In [ ]:
live_sales_doug_2017.columns = ["animal_sales_usd"]

In [ ]:
live_sales_doug_2017 = live_sales_doug_2017.reset_index()

In [ ]:
live_sales_doug_2017["Year"] = 2017

### Net Farm Income

In [ ]:
net_cash_income = nass_clean(pd.read_csv("E:/DISES/quantitative_data/income_net_cash_farm_usd.csv"))

In [ ]:
net_cash_income["Data Item"].unique()

In [ ]:
net_cash_income["Year"].unique()

### Direct Sales 

In [ ]:
direct_sales_retail_2017 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/direct_sale_retail_2017_22.csv"))
direct_sales_wholesale_2017 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/direct_sale_wholesale_2017_22.csv"))

In [ ]:
direct_sales_retail_2017["Data Item"].unique()

In [ ]:
direct_sales_retail_2017["Year"].unique()

In [ ]:
direct_sales_wholesale_2017["Data Item"].unique()

### Urban - rural continuum

In [ ]:
urban_rural = pd.read_excel("E:/DISES/quantitative_data/urban_rural_continuum_codes_2013.xls")

In [ ]:
urban_rural["GEOID"] = urban_rural["FIPS"].apply(lambda x: str(x).zfill(5))

In [ ]:
urban_rural

In [ ]:
urban_rural["RUCC_2013"].unique()

### Full owner, part owner and tenant percent

In [ ]:
owner_full_part_tenant_97 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_acres_tenure_owner_based_1997.csv"))

In [ ]:
owner_full_part_tenant_02_22 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_acres_tenure_owner_based.csv"))

In [ ]:
owner_full_part_tenant = pd.concat([owner_full_part_tenant_97, owner_full_part_tenant_02_22], axis = 0)

In [ ]:
owner_full_part_tenant.head()

In [ ]:
owner_full_part_tenant["Year"].unique()

In [ ]:
owner_full_part_tenant_pivot = owner_full_part_tenant.pivot_table(values="Value", index = ["GEOID", "Year"], columns="Domain Category")

In [ ]:
owner_full_part_tenant_pivot.columns = ["full_owner", 'part_owner', "tenant"]

In [ ]:
owner_full_part_tenant_pivot.loc[owner_full_part_tenant_pivot["tenant"].isna(), "tenant"] = 0

In [ ]:
owner_full_part_tenant_pivot = owner_full_part_tenant_pivot.div(owner_full_part_tenant_pivot.sum(axis=1), axis=0).reset_index()

In [ ]:
#making a new columns that is total rented %, which is used 
owner_full_part_tenant_pivot["rented_fract"] = owner_full_part_tenant_pivot["part_owner"]+owner_full_part_tenant_pivot["tenant"]

In [ ]:
owner_full_part_tenant_pivot.head()

### Farm operations with sales

In [ ]:
ops_with_sales_1997 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_with_sales_1997.csv"))
ops_with_sales_2002 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_with_sales_2002.csv"))
ops_with_sales_2007 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_with_sales_2007.csv"))
ops_with_sales_2012 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_with_sales_2012.csv"))
ops_with_sales_2017 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_with_sales_2017.csv"))
ops_with_sales_2022 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_operations_with_sales_2022.csv"))

In [ ]:
ops_with_sales_all = pd.concat([ops_with_sales_1997, ops_with_sales_2002,ops_with_sales_2007,
                                ops_with_sales_2012,ops_with_sales_2017,ops_with_sales_2022], axis = 0)

In [ ]:
ops_with_sales_all_pivot = ops_with_sales_all.pivot_table(values = "Value", index = ["Year", "GEOID"], columns="Domain Category")
ops_with_sales_all_pivot

In [ ]:
ops_with_sales_all_lt_10k = ops_with_sales_all_pivot[["FARM SALES: (1,000 TO 2,499 $)",
                                           "FARM SALES: (2,500 TO 4,999 $)",
                                           "FARM SALES: (5,000 TO 9,999 $)"]]

In [ ]:
ops_with_sales_all_lt_10k = pd.DataFrame(ops_with_sales_all_lt_10k.sum(axis = 1), columns=['no_ops_sales_lt_10k']).reset_index()

In [ ]:
ops_with_sales_all_lt_10k

In [ ]:
ops_with_sales_all_lt_250k = ops_with_sales_all_pivot[["FARM SALES: (1,000 TO 2,499 $)",
                                           "FARM SALES: (2,500 TO 4,999 $)",
                                           "FARM SALES: (5,000 TO 9,999 $)",
                                                     "FARM SALES: (10,000 TO 19,999 $)",
                                                     "FARM SALES: (20,000 TO 24,999 $)",
                                                     "FARM SALES: (25,000 TO 39,999 $)",
                                                     "FARM SALES: (40,000 TO 49,999 $)",
                                                     "FARM SALES: (50,000 TO 99,999 $)",
                                                     "FARM SALES: (100,000 TO 249,999 $)"]]

In [ ]:
ops_with_sales_all_lt_250k = pd.DataFrame(ops_with_sales_all_lt_250k.sum(axis = 1), columns=['no_ops_sales_lt_250k']).reset_index()

In [ ]:
ops_with_sales_all_gt_250k = ops_with_sales_all_pivot[["FARM SALES: (250,000 TO 499,999 $)",
                                           "FARM SALES: (500,000 OR MORE $)"
                                           ]]

In [ ]:
ops_with_sales_all_gt_250k = pd.DataFrame(ops_with_sales_all_gt_250k.sum(axis = 1), columns=['no_ops_sales_gt_250k']).reset_index()

In [ ]:
ops_with_sales_all_gt_500k = ops_with_sales_all_pivot[[
                                           "FARM SALES: (500,000 OR MORE $)"
                                           ]]

In [ ]:
ops_with_sales_all_gt_500k = pd.DataFrame(ops_with_sales_all_gt_500k.sum(axis = 1), columns=['no_ops_sales_gt_500k']).reset_index()

### Sales by sales categories (% of sales by  mega farms (> 500,000 USD)

In [ ]:
sales_by_category_1997 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_by_sales_categories_usd_1997.csv"))
sales_by_category_2002 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_by_sales_categories_usd_2002.csv"))
sales_by_category_2007 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_by_sales_categories_usd_2007.csv"))
sales_by_category_2017 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_by_sales_categories_usd_2017.csv"))
sales_by_category_2012 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_by_sales_categories_usd_2012.csv"))
sales_by_category_2022 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_by_sales_categories_usd_2022.csv"))

In [ ]:
sales_by_category_2017

In [ ]:
ops_with_sales_2007

In [ ]:
sales_by_category_all = pd.concat([sales_by_category_1997, sales_by_category_2002,
                                   sales_by_category_2012, sales_by_category_2017,
                                  sales_by_category_2022], axis = 0)

In [ ]:
sales_by_category_all["Domain Category"].unique()

In [ ]:
sales_by_category_all_gt_500k = sales_by_category_all[sales_by_category_all["Domain Category"]=="FARM SALES: (500,000 OR MORE $)"]
sales_by_category_all_gt_500k = sales_by_category_all_gt_500k.reset_index(drop = True)
sales_by_category_all_gt_500k["Data Item"] = "FARM SALES: (500,000 OR MORE $)"
sales_by_category_all_gt_500k

### Demographics

In [ ]:
farm_acres_size_1997 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_acres_operated_by_size_1997.csv"))
farm_acres_size_2002 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_acres_operated_by_size_2002.csv"))
farm_acres_size_2007 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_acres_operated_by_size_2007.csv"))
farm_acres_size_2012 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_acres_operated_by_size_2012.csv"))
farm_acres_size_2017 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_acres_operated_by_size_2017.csv"))
farm_acres_size_2022 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/farm_acres_operated_by_size_2022.csv"))

In [ ]:
farm_acres_size = pd.concat([farm_acres_size_1997, farm_acres_size_2002, farm_acres_size_2007,
          farm_acres_size_2012, farm_acres_size_2007, farm_acres_size_2022], axis = 0)

In [ ]:
farm_acres_size

In [ ]:
farm_acres_size["Year"].unique()

### Producers demographics

In [ ]:
producer_age_sex_race_2017 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/producer_gender_race_age_exp_2017.csv"))
producer_age_sex_race_2022 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/producer_gender_race_age_exp_2022.csv"))

In [ ]:
producer_age_sex_race_all = pd.concat([producer_age_sex_race_2017, producer_age_sex_race_2022], axis = 0)

In [ ]:
producer_age_sex_race_all.head()

In [ ]:
producer_age_sex_race_all["Domain"].unique()

In [ ]:
producer_age_sex_race_all = producer_age_sex_race_all.pivot_table(values="Value", columns="Data Item", index=["Year", "GEOID"]).reset_index()

In [ ]:
producer_age_sex_race_all.head()

In [ ]:
producer_age_sex_race_all.columns=["Year", "GEOID", "Producer_age", "Producer_experience_any_oper","Producer_experience_present_oper",
                                    "Producer_age_bw_25_34","Producer_age_bw_35_44", "Producer_age_le_35","Producer_age_lt_25",
                                   "Producer_black", "Producer_female", "Producer_male", "Producer_white"]
                                 

In [ ]:
producer_age_sex_race_all["Producer_total_no"] =producer_age_sex_race_all["Producer_male"] + producer_age_sex_race_all["Producer_female"] 

In [ ]:
producer_age_sex_race_all["Producer_age_lt_45"] = producer_age_sex_race_all["Producer_age_lt_25"] + \
producer_age_sex_race_all["Producer_age_bw_25_34"] + producer_age_sex_race_all["Producer_age_bw_35_44"]

In [ ]:
producer_age_sex_race_all.columns

In [ ]:
producer_age_sex_race_all = producer_age_sex_race_all.drop(['Producer_age_bw_25_34', 'Producer_age_bw_35_44', 
                                                  "Producer_age_lt_25","Producer_age_lt_45",
                                                 "Producer_age", 'Producer_black','Producer_male',
                                                 ],axis = 1)

In [ ]:
producer_age_sex_race_all["Producer_nonwhite"] = producer_age_sex_race_all["Producer_total_no"] - producer_age_sex_race_all["Producer_white"]

In [ ]:
producer_age_sex_race_all = producer_age_sex_race_all.drop("Producer_white", axis = 1)

In [ ]:
producer_age_sex_race_all.head()

In [ ]:
producer_age_sex_race_all[producer_age_sex_race_all["Producer_age_le_35"].isna()]["Year"].value_counts()

In [ ]:
producer_age_sex_race_all.isna().sum()

#### Beginning producers (experience <10) or le 11

In [ ]:
producer_exp_lt10 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/producer_beginner_exp_lt_10yr.csv"))

In [ ]:
producer_exp_lt10

### EQIP acres and rent

In [ ]:
eqip_df = pd.read_csv("E:/DISES/quantitative_data/eqip.csv")

In [ ]:
eqip_df.shape

In [ ]:
eqip_df["GEOID"] = eqip_df["GEOID"].astype("string").str.zfill(5)

In [ ]:
eqip_df = eqip_df[eqip_df["GEOID"].isin(us_county["GEOID"])]

In [ ]:
eqip_df.columns = ["GEOID", "Year", "EQIP_ACRE", "EQIP_RENT"]

In [ ]:
eqip_df["Year"].unique()

In [ ]:
eqip_df.head()

# Biofuel data (for corn and soybean)

In [ ]:
biofuel_corn = pd.read_csv(r"E:/DISES/quantitative_data/biofuel_corn_soy_limb_atel/Corn Ethanol/Corn Ethanol/ethanol_corn_use_by_county.csv")
biofuel_corn["GEOID"] = biofuel_corn["FIPS"].astype("string").str.zfill(5)

In [ ]:
biofuel_corn = biofuel_corn[["GEOID", "percent_used"]]  #limb et al, 2024

In [ ]:
biofuel_soy = pd.read_csv(r"E:/DISES/quantitative_data/biofuel_corn_soy_limb_atel/Soybean Biodiesel/Soybean Biodiesel/biodiesel_soybean_use_by_county.csv")
biofuel_soy["GEOID"] = biofuel_soy["FIPS"].astype("string").str.zfill(5)

biofuel_soy = biofuel_soy[["GEOID", "percent_used"]]  #limb et al, 2024

In [ ]:
biofuel_soy["Year"] = 2017   #they used 2017 USDA crop census data
biofuel_corn["Year"] = 2017   #they used 2017 USDA crop census data

In [ ]:
biofuel_corn = biofuel_corn.rename(columns={"percent_used":"biofuel_corn_percent_used"})
biofuel_soy = biofuel_soy.rename(columns={"percent_used":"biofuel_soy_percent_used"})

# CROP Price Received (state level)

In [ ]:
price_received = pd.read_csv("E:/DISES/quantitative_data/price_received_state.csv",low_memory=False)

price_received = price_received[~price_received["Value"].str.contains(r"X|S|D|NA", na=False)]

price_received = price_received[~price_received["State ANSI"].isna()]

In [ ]:
price_received_us = pd.read_csv("E:/DISES/quantitative_data/price_received_national.csv")
price_received_us = price_received_us[~price_received_us["Value"].str.contains(r"X|S|D|NA", na=False)]

In [ ]:
price_received["Value"] = price_received["Value"].astype("float")

In [ ]:
price_received_us["Value"] = price_received_us["Value"].astype("float")

In [ ]:
price_received_us["Period"].unique()

In [ ]:
price_received_monthly = price_received[price_received["Period"]!="MARKETING YEAR"]

In [ ]:
price_received_mrkt_yr = price_received[price_received["Period"]=="MARKETING YEAR"]
price_received_mrkt_yr = price_received_mrkt_yr[["Year", "State", "State ANSI", "Commodity", "Data Item", "Value"]]
price_received_mrkt_yr["Value"] = price_received_mrkt_yr["Value"].astype("float")
price_received_mrkt_yr["Commodity_col"] = price_received_mrkt_yr["Data Item"].str.split("-").str[0] +"("+ price_received_mrkt_yr["Data Item"].str.split("IN").str[-1] +")"

In [ ]:
price_received_mrkt_yr_pivot = price_received_mrkt_yr.pivot_table(values = "Value", index = ["Year", "State", "State ANSI"], columns="Commodity_col").reset_index()

In [ ]:
price_received_mrkt_yr_pivot1 = price_received_mrkt_yr_pivot.copy()

In [ ]:
for i in range(len(price_received_mrkt_yr_pivot1)):
    price_received_mrkt_yr_pivot1


In [ ]:
price_received_mrkt_yr_us = price_received_us[price_received_us["Period"]=="MARKETING YEAR"]
price_received_mrkt_yr_us = price_received_mrkt_yr_us[["Year", "State", "State ANSI", "Commodity", "Data Item", "Value"]]
price_received_mrkt_yr_us["Value"] = price_received_mrkt_yr_us["Value"].astype("float")
price_received_mrkt_yr_us["Commodity_col"] = price_received_mrkt_yr_us["Data Item"].str.split("-").str[0] +"("+ price_received_mrkt_yr_us["Data Item"].str.split("IN").str[-1] +")"

In [ ]:
crop_cols = price_received_mrkt_yr_pivot1.columns[3:]

In [ ]:
merged_price = price_received_mrkt_yr_pivot1.merge(
    price_received_mrkt_yr_pivot_us[["Year"] + list(crop_cols)],
    on="Year",
    how="left",
    suffixes=("", "_nat")  # '_nat' for national columns
)

# Fill missing state-level prices using national-level prices
for col in crop_cols:
    merged_price[col] = merged_price[col].fillna(merged_price[f"{col}_nat"])

# Drop the extra national columns
merged_price = merged_price[price_received_mrkt_yr_pivot1.columns]

# Result
price_received_mrkt_yr_pivot_na_filled = merged_price


In [ ]:
price_received_mrkt_yr_pivot_us = price_received_mrkt_yr_us.pivot_table(values = "Value", index = ["Year"], columns="Commodity_col").reset_index()

In [ ]:
price_received_mrkt_yr_pivot_na_filled

In [ ]:
price_received_mrkt_yr_pivot.to_csv("E:/DISES/quantitative_data/price_received_st_mrkt_yr.csv", index=False)

In [ ]:
price_received_mrkt_yr_pivot_na_filled.to_csv("E:/DISES/quantitative_data/price_received_st_mrkt_yr_na_filled_with_US.csv", index=False)

In [ ]:
price_received_mrkt_yr_pivot.dtypes


In [ ]:
# price_received_mrkt_yr_pivot_avg_08_19 = price_received_mrkt_yr_pivot[price_received_mrkt_yr_pivot["Year"].between(2008, 2019,inclusive = "both")] \
# .groupby(["State","State ANSI"]).mean().reset_index().drop("Year", axis = 1)

# price_received_mrkt_yr_pivot_avg_08_19

price_received_mrkt_yr_pivot_avg_08_19 = price_received_mrkt_yr_pivot_na_filled[price_received_mrkt_yr_pivot_na_filled["Year"].between(2008, 2019,inclusive = "both")] \
.groupby(["State","State ANSI"]).mean().reset_index().drop("Year", axis = 1)

price_received_mrkt_yr_pivot_avg_08_19

In [ ]:
price_received_mrkt_yr_pivot_avg_08_19_ct_gpd = gpd.GeoDataFrame(us_county.merge(price_received_mrkt_yr_pivot_avg_08_19, left_on="STATEFP", right_on="State ANSI", how="left"),
                                                                 geometry="geometry")   

In [ ]:
price_received_mrkt_yr_pivot_avg_08_19_ct_gpd_east = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[price_received_mrkt_yr_pivot_avg_08_19_ct_gpd["STATEFP"].isin(st_id_east_int)]

In [ ]:
price_received_mrkt_yr_pivot_avg_08_19_ct_gpd

In [ ]:
price_received_mrkt_yr_pivot_avg_08_19_ct_gpd.columns

In [ ]:
# Here states without data are filled with the national price
# BARLEY
price_received_mrkt_yr_pivot_avg_08_19_ct_barley = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "BARLEY ( $ / BU)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_barley.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_barley.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_BARLEY.csv", index=False)

# CORN
price_received_mrkt_yr_pivot_avg_08_19_ct_corn = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "CORN, GRAIN ( $ / BU)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_corn.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_corn.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_CORN.csv", index=False)

# COTTON
price_received_mrkt_yr_pivot_avg_08_19_ct_cotton = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "COTTON, UPLAND ( $ / LB)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_cotton.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_cotton.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_COTTON.csv", index=False)

# HAY
price_received_mrkt_yr_pivot_avg_08_19_ct_hay = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "HAY, (EXCL ALFALFA) ( $ / TON)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_hay.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_hay.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_HAY.csv", index=False)

# OATS
price_received_mrkt_yr_pivot_avg_08_19_ct_oats = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "OATS ( $ / BU)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_oats.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_oats.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_OATS.csv", index=False)

# PEANUTS
price_received_mrkt_yr_pivot_avg_08_19_ct_peanuts = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "PEANUTS ( $ / LB)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_peanuts.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_peanuts.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_PEANUTS.csv", index=False)

# PECANS
price_received_mrkt_yr_pivot_avg_08_19_ct_pecans = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "PECANS ( $ / LB)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_pecans.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_pecans.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_PECANS.csv", index=False)

# RYE
price_received_mrkt_yr_pivot_avg_08_19_ct_rye = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "RYE ( $ / BU)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_rye.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_rye.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_RYE.csv", index=False)

# SORGHUM
price_received_mrkt_yr_pivot_avg_08_19_ct_sorghum = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "SORGHUM, GRAIN ( $ / CWT)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_sorghum.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_sorghum.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_SORGHUM.csv", index=False)

# SOYBEANS
price_received_mrkt_yr_pivot_avg_08_19_ct_soybeans = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "SOYBEANS ( $ / BU)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_soybeans.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_soybeans.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_SOYBEANS.csv", index=False)

# WHEAT
price_received_mrkt_yr_pivot_avg_08_19_ct_wheat = price_received_mrkt_yr_pivot_avg_08_19_ct_gpd[["GEOID", "ST", "WHEAT, WINTER ( $ / BU)"]]
price_received_mrkt_yr_pivot_avg_08_19_ct_wheat.columns = ["GEOID", "ST", "PRICE"]
price_received_mrkt_yr_pivot_avg_08_19_ct_wheat.to_csv("E:/DISES/quantitative_data/price_received_mrkt_yr_pivot_avg_08_19_ct_WHEAT.csv", index=False)


# Asset value (land included) farm real estate value

In [ ]:
real_estate_value = nass_clean(pd.read_csv("E:/DISES/quantitative_data/ag_land_asset_value.csv"))

In [ ]:
real_estate_value

# Labor counts(number of laborers)

In [ ]:
labor_number = nass_clean(pd.read_csv("E:/DISES/quantitative_data/labor_number_hired_migrant_unpaid.csv"), program="census")

In [ ]:
labor_number["Data Item"].unique()

In [ ]:
#summing all kind of labor numbers
labor_number = labor_number.drop(["Data Item"], axis = 1).groupby(["Year", "State", "State ANSI","County ANSI", "County", "Commodity", "GEOID"]).sum().reset_index()
labor_number["Data Item"] = "LABOR NUMBER, HIRED, MIGRANT, UNPAID"

In [ ]:
labor_number

# Standardization variable

In [ ]:
total_cropland = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_cropland_acres.csv"))

In [ ]:
total_cropland

In [ ]:
total_farmland_97 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_farmland_acres_operated_1997.csv"))
total_farmland_02 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_farmland_acres_operated_2002.csv"))
total_farmland_07 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_farmland_acres_operated_2007.csv"))
total_farmland_12 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_farmland_acres_operated_2012.csv"))
total_farmland_17 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_farmland_acres_operated_2017.csv"))
total_farmland_22 = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_farmland_acres_operated_2022.csv"))

In [ ]:
total_farmland_02

In [ ]:
total_farmland_all = pd.concat([total_farmland_97, total_farmland_02,total_farmland_07, total_farmland_12, total_farmland_17, total_farmland_22],
          axis = 0)

In [ ]:
total_farmland_all = total_farmland_all[total_farmland_all["Domain"] == "TOTAL"]

In [ ]:
total_number_farm_op = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_number_operations.csv"))

In [ ]:
total_sales_usd = nass_clean(pd.read_csv("E:/DISES/quantitative_data/total_sales_commodity_usd.csv"))

In [ ]:
crop_sales_usd = nass_clean(pd.read_csv("E:/DISES/quantitative_data/crop_sales_usd.csv"))

In [ ]:
df_standardize = pd.concat([total_cropland, total_farmland_all, total_number_farm_op, total_sales_usd, crop_sales_usd], axis= 0)

In [ ]:
df_standardize_pivot = df_standardize.pivot_table(values="Value", index = ["Year","GEOID"], columns="Data Item")

In [ ]:
df_standardize_pivot

In [ ]:
df_standardize_pivot.columns = ["tot_cropland_acre", "tot_sales_usd", "crop_sales_usd", "tot_farm_acre", "tot_no_operation"]

In [ ]:
df_standardize_pivot = df_standardize_pivot.reset_index()

In [ ]:
df_standardize_pivot

# Combining all to single dataframe

In [ ]:
df_combine1 = pd.concat([irg_acre_total, cover_crop_acre, till_no_till_acre, till_conservational_acre, till_conventional_acre,
            organic_number_operations, precision_ag_number_operations, tile_drain_acres,
           crop_insurance_acres, crop_animal_insur_usd, crp_prog_acres, govt_prog_all_receipt_usd,govt_prog_all_cons_wet_usd,
           fert_exp_usd, chem_exp_usd, labor_usd, machinery_usd, live_cattle_inventory, live_cattle_sales,
           live_goat_inventory, live_goat_sales, live_hogs_inventory, live_hogs_sales, live_sheep_inventory, live_sheep_sales,
                        median_farm_size, crp_acre_fsa_melt, crp_usd_fsa_melt, 
                         direct_sales_wholesale_2017,direct_sales_retail_2017, sales_by_category_all_gt_500k,
                        producer_exp_lt10, net_cash_income,
                        labor_number, real_estate_value],  #these two are new vars (common for all crops)
          axis = 0)
           

In [ ]:
df_combine1_pivot = df_combine1.pivot_table(values="Value", columns="Data Item", index=["Year", "GEOID"])

In [ ]:
df_combine1_pivot.columns

In [ ]:
#note: there was an issue in prpevious code- fert_exp_usd and sales_usd_gt_500k were interchanged by mistake
df_combine1_pivot.columns = ["crop_insurance_acres", "irg_acre_total", "ag_land_value_usd" , "live_cattle_inventory",
                           "live_cattle_sales", "chem_exp_usd", "directsales_retail", "directsales_wholesale",
                             'CRP_ACRE', 'CRP_USD', "organic_number_operations", "sales_usd_gt_500k","fert_exp_usd", 
                         "live_goat_inventory", "live_goat_sales", "govt_prog_all_receipt_usd", "conserv_prog_acres",
                        "govt_prog_all_cons_wet_usd", "live_hogs_inventory", "live_hogs_sales", "crop_animal_insur_usd",
                        "net_cash_income","labor_number","labor_usd", "machinery_usd", "median_farmsize", "till_conservational_acre",
                             "till_no_till_acre","till_conventional_acre","cover_crop_acre", 
                             "tile_drain_acres", "precision_ag_number_operations", 
                           "producer_exp_lt_10", "live_sheep_inventory", "live_sheep_sales"
                        
                           ]

In [ ]:
df_combine1_pivot = df_combine1_pivot.reset_index()

In [ ]:
# combining no till and conventional till toghther
#note- see if this increases NaNs; if it does, use df_combine1_pivot[["till_no_till_acre", "till_conventional_acre"]].sum(axis = 1)

df_combine1_pivot["till_new_conservational_acre"] = df_combine1_pivot["till_no_till_acre"] +df_combine1_pivot["till_conservational_acre"]
df_combine1_pivot = df_combine1_pivot.drop(["till_no_till_acre", "till_conservational_acre"], axis = 1)

In [ ]:
df_combine1_pivot.head()

In [ ]:
#dropping these columns as using Farm service data on crp acres and rent
df_combine1_pivot = df_combine1_pivot.drop(["govt_prog_all_receipt_usd", "conserv_prog_acres", "govt_prog_all_cons_wet_usd"], axis = 1)

In [ ]:
df_combine1_pivot.shape

In [ ]:
df_combine2_pivot = df_combine1_pivot.merge(NP_from_manure_fert, on = ["GEOID", "Year"], how = 'outer')

In [ ]:
df_combine2_pivot

In [ ]:
df_combine3_pivot = df_combine2_pivot.merge(producer_age_sex_race_all, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine3_pivot.shape

In [ ]:
df_combine4_pivot = df_combine3_pivot.merge(urban_rural[["GEOID", "RUCC_2013"]], on = "GEOID")

In [ ]:
df_combine5_pivot = df_combine4_pivot.merge(owner_full_part_tenant_pivot[["GEOID","Year","rented_fract"]], on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine6_pivot = df_combine5_pivot.merge(df_standardize_pivot, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine6_pivot.columns

In [ ]:
ops_with_sales_all_lt_10k

In [ ]:
df_combine7_pivot = df_combine6_pivot.merge(ops_with_sales_all_lt_10k, on = ["Year", "GEOID"], how = "outer")

In [ ]:
# df_combine7_pivot = df_combine7_pivot.merge(ops_with_sales_all_gt_500k, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine7_pivot.head()

In [ ]:
live_sales_doug_2017

In [ ]:
df_combine8_pivot = df_combine7_pivot.merge(live_sales_doug_2017, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine9_pivot = df_combine8_pivot.merge(eqip_df, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine10_pivot = df_combine9_pivot.merge(h2a_worker, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine11_pivot = df_combine10_pivot.merge(biofuel_corn, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine12_pivot = df_combine11_pivot.merge(biofuel_soy, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine13_pivot = df_combine12_pivot.merge(crop_insurance_acres_rma, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine14_pivot = df_combine13_pivot.merge(age_megatrend_pivot, on = ["Year", "GEOID"], how = "outer")

In [ ]:
df_combine14_pivot.columns

df_combine_raw_final is the unstandardized original values of all the agricultural variabls

In [ ]:
# This include all year data
df_combine_raw_final = df_combine14_pivot.copy()

In [ ]:
df_combine_raw_final.head()

In [ ]:
#Standardizing the raw averaged dataframe
df_combine_raw_final_std = df_combine_raw_final.copy()

#removinf variables which are no longer used
df_combine_raw_final_std = df_combine_raw_final_std.drop(["Producer_experience_present_oper","Producer_experience_any_oper",
                                                                      "precision_ag_number_operations",
                                                          
                                                                     "crop_insurance_acres",
                                                                     "till_conventional_acre",
                                                                     "crop_animal_insur_usd",
                            "live_cattle_sales", "live_hogs_sales", "live_sheep_sales", "live_goat_sales",
                                          "live_hogs_inventory", "live_sheep_inventory", "live_goat_inventory",
                                 "CRP_USD", "directsales_wholesale","EQIP_RENT",
                                   "no_h2a_workers","net_cash_income"], axis = 1)

df_combine_raw_final_std["irg_acre_total"] = df_combine_raw_final_std["irg_acre_total"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["live_cattle_inventory"] = df_combine_raw_final_std["live_cattle_inventory"]/df_combine_raw_final_std["tot_farm_acre"]
df_combine_raw_final_std["chem_exp_usd"] = df_combine_raw_final_std["chem_exp_usd"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["directsales_retail"] = df_combine_raw_final_std["directsales_retail"]/df_combine_raw_final_std["tot_sales_usd"]
df_combine_raw_final_std["CRP_ACRE"] = df_combine_raw_final_std["CRP_ACRE"]/df_combine_raw_final_std["tot_farm_acre"]
df_combine_raw_final_std["organic_number_operations"] = df_combine_raw_final_std["organic_number_operations"]/df_combine_raw_final_std["tot_no_operation"]
df_combine_raw_final_std["fert_exp_usd"] = df_combine_raw_final_std["fert_exp_usd"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["machinery_usd"] = df_combine_raw_final_std["machinery_usd"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["median_farmsize"] = df_combine_raw_final_std["median_farmsize"]
df_combine_raw_final_std["till_new_conservational_acre"] = df_combine_raw_final_std["till_new_conservational_acre"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["cover_crop_acre"] = df_combine_raw_final_std["cover_crop_acre"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["tile_drain_acres"] = df_combine_raw_final_std["tile_drain_acres"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["N_fert_kg"] = df_combine_raw_final_std["N_fert_kg"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["P_fert_kg"] = df_combine_raw_final_std["P_fert_kg"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["N_manure_kg"] = df_combine_raw_final_std["N_manure_kg"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["P_manure_kg"] = df_combine_raw_final_std["P_manure_kg"]/df_combine_raw_final_std["tot_cropland_acre"]
df_combine_raw_final_std["Producer_female"] = df_combine_raw_final_std["Producer_female"]/df_combine_raw_final_std["Producer_total_no"]
df_combine_raw_final_std["Producer_nonwhite"] = df_combine_raw_final_std["Producer_nonwhite"]/df_combine_raw_final_std["Producer_total_no"]
df_combine_raw_final_std["Producer_age_le_35"] = df_combine_raw_final_std["Producer_age_le_35"]/df_combine_raw_final_std["Producer_total_no"]
df_combine_raw_final_std["rented_fract"] = df_combine_raw_final_std["rented_fract"]
df_combine_raw_final_std["RUCC_2013"] = df_combine_raw_final_std["RUCC_2013"]
df_combine_raw_final_std["animal_sales_usd"] = df_combine_raw_final_std["animal_sales_usd"]/df_combine_raw_final_std["tot_sales_usd"]
df_combine_raw_final_std["no_ops_sales_lt_10k"] = df_combine_raw_final_std["no_ops_sales_lt_10k"]/df_combine_raw_final_std["tot_no_operation"]
df_combine_raw_final_std["sales_usd_gt_500k"] = df_combine_raw_final_std["sales_usd_gt_500k"]/df_combine_raw_final_std["tot_sales_usd"]

df_combine_raw_final_std["producer_exp_lt_10"] = df_combine_raw_final_std["producer_exp_lt_10"]/df_combine_raw_final_std["Producer_total_no"]

df_combine_raw_final_std["EQIP_ACRE"] = df_combine_raw_final_std["EQIP_ACRE"]/df_combine_raw_final_std["tot_farm_acre"]
df_combine_raw_final_std["biofuel_corn_percent_used"] = df_combine_raw_final_std["biofuel_corn_percent_used"]
df_combine_raw_final_std["biofuel_soy_percent_used"] = df_combine_raw_final_std["biofuel_soy_percent_used"]
df_combine_raw_final_std["crop_insurance_acre_rma"] = df_combine_raw_final_std["crop_insurance_acre_rma"]/df_combine_raw_final_std["tot_cropland_acre"]

df_combine_raw_final_std["prod_oper_age_le_35_std"] = df_combine_raw_final_std["prod_oper_age_le_35_std"]
df_combine_raw_final_std["prod_oper_age"] = df_combine_raw_final_std["prod_oper_age"]

df_combine_raw_final_std["ag_land_value_usd"] = df_combine_raw_final_std["ag_land_value_usd"]/df_combine_raw_final_std["tot_farm_acre"]
df_combine_raw_final_std["labor_number"] = df_combine_raw_final_std["labor_number"]/df_combine_raw_final_std["tot_cropland_acre"]


In [ ]:
#  fixing RMA crop insurance acre > cropland acres 
df_combine_raw_final_std.loc[df_combine_raw_final_std["crop_insurance_acre_rma"]>1, "crop_insurance_acre_rma"] = 1

df_combine_raw_final_std.loc[df_combine_raw_final_std["animal_sales_usd"]>1, "animal_sales_usd"] = 1

#at 1-2 instances, few variable std value >1:
# df_combine_raw_final_std.loc[df_combine_raw_final_std["till_no_till_acre"]>1, "till_no_till_acre"] = 1
df_combine_raw_final_std.loc[df_combine_raw_final_std["till_new_conservational_acre"]>1, "till_new_conservational_acre"] = 1
df_combine_raw_final_std.loc[df_combine_raw_final_std["tile_drain_acres"]>1, "tile_drain_acres"] = 1
df_combine_raw_final_std.loc[df_combine_raw_final_std["sales_usd_gt_500k"]>1, "sales_usd_gt_500k"] = 1
df_combine_raw_final_std.loc[df_combine_raw_final_std["EQIP_ACRE"]>1, "EQIP_ACRE"] = 1

In [ ]:
df_combine_raw_final_std.columns

In [ ]:
df_combine_raw_final_std.shape

In [ ]:
df_combine_raw_final.columns

In [ ]:
# Only taking years 2012 and 2017
# df_combine_raw_final_12_17 = df_combine_raw_final[df_combine_raw_final["Year"].isin([2012, 2017])]
df_combine_raw_final_12_17 = df_combine_raw_final[df_combine_raw_final["Year"].between(left = 2009, right =2018)]
df_combine_raw_final_12_17 = df_combine_raw_final_12_17.reset_index(drop = True)

In [ ]:
df_combine_raw_final_12_17.isna().all()

This shows organic acres and precision ag acres are not available for 2012 and 2017, they have all NA values

In [ ]:
#dropping columns which have all NaN values
df_combine_raw_final_12_17 = df_combine_raw_final_12_17.dropna(axis = 1, how = "all")

In [ ]:
df_combine_raw_final_12_17

In [ ]:
# This is the average over the time periods
df_combine_raw_final_12_17_avg = df_combine_raw_final_12_17.groupby(["GEOID"]).mean().drop("Year", axis = 1).reset_index()

In [ ]:
#removinf variables which are no longer used
df_combine_raw_final_12_17_avg = df_combine_raw_final_12_17_avg.drop(["Producer_experience_present_oper",
                                                                      "Producer_experience_any_oper",
                                                                     "crop_insurance_acres",
                                                                     "till_conventional_acre",
                                                                     "crop_animal_insur_usd",
                            "live_cattle_sales", "live_hogs_sales", "live_sheep_sales", "live_goat_sales",
                                          "live_hogs_inventory", "live_sheep_inventory", "live_goat_inventory",
                                 "CRP_USD", "directsales_wholesale","EQIP_RENT",
                                   "no_h2a_workers","net_cash_income",
                                    "prod_oper_age", "prod_oper_age_le_35_std"], axis = 1)

In [ ]:
df_combine_raw_final_12_17_avg.columns

In [ ]:
df_combine_raw_final_12_17_avg.shape

## REMINDER- NOTE: "Producer_experience_any_oper"


In [ ]:
#Standardizing the raw averaged dataframe
df_combine_std_12_17_final = df_combine_raw_final_12_17_avg.copy()

df_combine_std_12_17_final["irg_acre_total"] = df_combine_std_12_17_final["irg_acre_total"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["live_cattle_inventory"] = df_combine_std_12_17_final["live_cattle_inventory"]/df_combine_std_12_17_final["tot_farm_acre"]
df_combine_std_12_17_final["chem_exp_usd"] = df_combine_std_12_17_final["chem_exp_usd"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["directsales_retail"] = df_combine_std_12_17_final["directsales_retail"]/df_combine_std_12_17_final["tot_sales_usd"]
df_combine_std_12_17_final["CRP_ACRE"] = df_combine_std_12_17_final["CRP_ACRE"]/df_combine_std_12_17_final["tot_farm_acre"]
df_combine_std_12_17_final["organic_number_operations"] = df_combine_std_12_17_final["organic_number_operations"]/df_combine_std_12_17_final["tot_no_operation"]
df_combine_std_12_17_final["fert_exp_usd"] = df_combine_std_12_17_final["fert_exp_usd"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["labor_usd"] = df_combine_std_12_17_final["labor_usd"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["machinery_usd"] = df_combine_std_12_17_final["machinery_usd"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["median_farmsize"] = df_combine_std_12_17_final["median_farmsize"]
df_combine_std_12_17_final["till_new_conservational_acre"] = df_combine_std_12_17_final["till_new_conservational_acre"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["cover_crop_acre"] = df_combine_std_12_17_final["cover_crop_acre"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["tile_drain_acres"] = df_combine_std_12_17_final["tile_drain_acres"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["N_fert_kg"] = df_combine_std_12_17_final["N_fert_kg"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["P_fert_kg"] = df_combine_std_12_17_final["P_fert_kg"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["N_manure_kg"] = df_combine_std_12_17_final["N_manure_kg"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["P_manure_kg"] = df_combine_std_12_17_final["P_manure_kg"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["Producer_female"] = df_combine_std_12_17_final["Producer_female"]/df_combine_std_12_17_final["Producer_total_no"]
df_combine_std_12_17_final["Producer_nonwhite"] = df_combine_std_12_17_final["Producer_nonwhite"]/df_combine_std_12_17_final["Producer_total_no"]
df_combine_std_12_17_final["Producer_age_le_35"] = df_combine_std_12_17_final["Producer_age_le_35"]/df_combine_std_12_17_final["Producer_total_no"]
df_combine_std_12_17_final["rented_fract"] = df_combine_std_12_17_final["rented_fract"]
df_combine_std_12_17_final["RUCC_2013"] = df_combine_std_12_17_final["RUCC_2013"]
df_combine_std_12_17_final["animal_sales_usd"] = df_combine_std_12_17_final["animal_sales_usd"]/df_combine_std_12_17_final["tot_sales_usd"]
df_combine_std_12_17_final["no_ops_sales_lt_10k"] = df_combine_std_12_17_final["no_ops_sales_lt_10k"]/df_combine_std_12_17_final["tot_no_operation"]
df_combine_std_12_17_final["sales_usd_gt_500k"] = df_combine_std_12_17_final["sales_usd_gt_500k"]/df_combine_std_12_17_final["tot_sales_usd"]

df_combine_std_12_17_final["producer_exp_lt_10"] = df_combine_std_12_17_final["producer_exp_lt_10"]/df_combine_std_12_17_final["Producer_total_no"]

df_combine_std_12_17_final["EQIP_ACRE"] = df_combine_std_12_17_final["EQIP_ACRE"]/df_combine_std_12_17_final["tot_farm_acre"]
df_combine_std_12_17_final["biofuel_corn_percent_used"] = df_combine_std_12_17_final["biofuel_corn_percent_used"]
df_combine_std_12_17_final["biofuel_soy_percent_used"] = df_combine_std_12_17_final["biofuel_soy_percent_used"]
df_combine_std_12_17_final["crop_insurance_acre_rma"] = df_combine_std_12_17_final["crop_insurance_acre_rma"]/df_combine_std_12_17_final["tot_cropland_acre"]
df_combine_std_12_17_final["ag_land_value_usd"] = df_combine_std_12_17_final["ag_land_value_usd"]/df_combine_std_12_17_final["tot_farm_acre"]
df_combine_std_12_17_final["labor_number"] = df_combine_std_12_17_final["labor_number"]/df_combine_std_12_17_final["tot_cropland_acre"]

In [ ]:
#  fixing RMA crop insurance acre > cropland acres 
df_combine_std_12_17_final.loc[df_combine_std_12_17_final["crop_insurance_acre_rma"]>1, "crop_insurance_acre_rma"] = 1

df_combine_std_12_17_final.loc[df_combine_std_12_17_final["animal_sales_usd"]>1, "animal_sales_usd"] = 1

#at 1-2 instances, few variable std value >1:
# df_combine_std_12_17_final.loc[df_combine_std_12_17_final["till_no_till_acre"]>1, "till_no_till_acre"] = 1
df_combine_std_12_17_final.loc[df_combine_std_12_17_final["till_new_conservational_acre"]>1, "till_new_conservational_acre"] = 1
df_combine_std_12_17_final.loc[df_combine_std_12_17_final["tile_drain_acres"]>1, "tile_drain_acres"] = 1
df_combine_std_12_17_final.loc[df_combine_std_12_17_final["sales_usd_gt_500k"]>1, "sales_usd_gt_500k"] = 1
df_combine_std_12_17_final.loc[df_combine_std_12_17_final["EQIP_ACRE"]>1, "EQIP_ACRE"] = 1


<span style="color:brown;"> Not including general NAICS info; crop specific NAICS information is added later while making crop specific panel </span>

In [ ]:
# df_combine_std_12_17_final = df_combine_std_12_17_final.merge(naics_std, on = "GEOID", how="left")

In [ ]:
df_combine_std_12_17_final[["GEOID", "tot_cropland_acre"]].dropna()

In [ ]:
df_combine_std_12_17_final[["GEOID", "tot_cropland_acre"]].to_csv("E:/DISES/quantitative_data/GEOID_tot_cropland.csv", index = None)

In [ ]:
df_combine_std_12_17_final = df_combine_std_12_17_final.drop(['tot_cropland_acre', 'tot_sales_usd','crop_sales_usd', 'tot_farm_acre',
       'tot_no_operation','Producer_total_no'], axis = 1)

In [ ]:
df_combine_std_12_17_final.columns

In [ ]:
df_combine_std_12_17_final

### Saving to CSV

In [ ]:
df_combine_raw_final.shape, df_combine_raw_final_12_17.shape, df_combine_raw_final_12_17_avg.shape, df_combine_std_12_17_final.shape


In [ ]:
df_combine_raw_final["Year"].unique()

In [ ]:
# FOR model 2.0

# 12_17 does not mean it only from 2012 to 2017; just coa variables are for 12-17 
# but other ag vars that has values from 2009 area also included

df_combine_raw_final_12_17.to_csv("E:/DISES/quantitative_data/data_input_RF_2//ag_data_2012_17.csv", index = None) #2012,17

df_combine_raw_final_12_17_avg.to_csv("E:/DISES/quantitative_data/data_input_RF_2//ag_data_2012_17_average.csv", index = None)

df_combine_std_12_17_final.to_csv("E:/DISES/quantitative_data/data_input_RF_2//ag_data_std_2012_17.csv", index = None) #standardized

In [ ]:
st_id_east = ["23", "38", "46" ,"55", "50", "27", "33" ,"19" ,"25" ,"31" ,"36" ,"42", "09", "44", "34", "18", "39", "17" ,"11", "10", "54", "24", "21" ,"20" ,"51", "29", "40" ,"37",
"47", "48" ,"01", "28", "13" ,"45", "05", "22", "12", "26"]

In [ ]:
len(st_id_east)

In [ ]:
df_combine_std_12_17_final_east = df_combine_std_12_17_final[df_combine_std_12_17_final["GEOID"].str[:2].isin(st_id_east)]

In [ ]:
df_combine_raw_final_12_17_avg_east = df_combine_raw_final_12_17_avg[df_combine_raw_final_12_17_avg["GEOID"].str[:2].isin(st_id_east)]